# CEOAI Practice 2 - Panda MNIST Minimum Solution

Objective: train two tiny Torch models and package a valid submission:

1. Load scanner arrays from `train_data.zip`.
2. Train `model_sub1.pt` for `(N, 1, 28, 28)` inputs.
3. Train `model_sub2.pt` for `(N, 3, 28, 28)` inputs.
4. Zip both TorchScript models into `submission.zip`.

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
torch.manual_seed(0)

In [ ]:
def ensure_unzipped(zip_path, target_dir):
    if target_dir.exists():
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir)

ensure_unzipped(DATA / "train_data.zip", DATA / "train_data")
ensure_unzipped(DATA / "test_data.zip", DATA / "test_data")

In [ ]:
def load_scanners(subtask, scanners):
    X_parts, y_parts, scanner_parts = [], [], []
    folder = DATA / "train_data" / subtask
    for scanner in scanners:
        X = np.load(folder / f"scanner{scanner}_X.npy")
        y = np.load(folder / f"scanner{scanner}_y.npy")
        X_parts.append(X)
        y_parts.append(y)
        scanner_parts.extend([scanner] * len(y))
    X = torch.tensor(np.concatenate(X_parts), dtype=torch.float32) / 255.0
    y = torch.tensor(np.concatenate(y_parts), dtype=torch.long)
    return X, y, np.array(scanner_parts)

X1, y1, scanners1 = load_scanners("subtask1", range(1, 4))
X2, y2, scanners2 = load_scanners("subtask2", range(1, 9))
print(X1.shape, y1.shape, X2.shape, y2.shape)

In [ ]:
class TinyDigitNet(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((7, 7))
        self.fc = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

def train_model(X, y, in_channels, epochs=18):
    dataset = TensorDataset(X, y)
    n_val = max(20, len(dataset) // 5)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(0))
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128)
    model = TinyDigitNet(in_channels)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = model(xb).argmax(dim=1)
            correct += int((pred == yb).sum())
            total += len(yb)
    return model, correct / total

model1, acc1 = train_model(X1, y1, in_channels=1)
model2, acc2 = train_model(X2, y2, in_channels=3)
print({"fixture_val_acc_sub1": acc1, "fixture_val_acc_sub2": acc2})
print({"params_sub1": sum(p.numel() for p in model1.parameters()), "params_sub2": sum(p.numel() for p in model2.parameters())})

In [ ]:
model1.eval()
model2.eval()
scripted1 = torch.jit.trace(model1, torch.zeros(1, 1, 28, 28))
scripted2 = torch.jit.trace(model2, torch.zeros(1, 3, 28, 28))
scripted1.save(OUT / "model_sub1.pt")
scripted2.save(OUT / "model_sub2.pt")

with zipfile.ZipFile(OUT / "submission.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT / "model_sub1.pt", "model_sub1.pt")
    zf.write(OUT / "model_sub2.pt", "model_sub2.pt")

with zipfile.ZipFile(OUT / "submission.zip") as zf:
    assert sorted(zf.namelist()) == ["model_sub1.pt", "model_sub2.pt"]
print("wrote", OUT / "submission.zip")